In [61]:
import pandas as pd
import re
import unicodedata

In [62]:
df = pd.read_excel("../data/raw/eventos_chubut.xlsx")

In [63]:
df_ch = df.copy()

In [64]:
df_ch["actores_participantes"].unique().tolist()

['Sindicato de la Salud Pública',
 'No aplica',
 'No especificado',
 'Auxiliares de la educación',
 'Miembros de comunidades originarias',
 'Personal de la salud/Sisap',
 'Docentes',
 'Sisap',
 'Consejo de Bienestar Policial',
 'amigos/colegas/seguidores de su programa radial/bandas musicales/alumnos/vecinos de distintas organizaciones sociales',
 'Error',
 'Sindicato de Camioneros',
 'Trabajadores del servicio de recolección de residuos y de la Planta de Tratamiento',
 'Feriantes del barrio Perón',
 'Soeme',
 'Consejo de Bienestar Policial del Chubut',
 'Consejo de Bienestar Policial/Familia policial',
 'Estibadores',
 'Sindicato de la Salud Pública (Sisap)',
 'Policías/Familias de policías',
 'Policía',
 'Familia de Maximiliano Curiqueo',
 'Policía activo y pasivo',
 'Vecinos de la Meseta del Chubut/Intendentes de la región/Referentes sociales y de comunidades originarias/Representantes gremiales y trabajadores de las seccionales Puerto Madryn/Paso de Indios y Esquel de la Unión Obre

In [65]:
df_ch = df_ch[~df_ch["actores_participantes"].isin(["Error", "No aplica"])].copy()

In [66]:
col_actores_ch = "actores_participantes"

In [67]:
print(col_actores_ch)

actores_participantes


In [68]:
def normalizar(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return texto

df_ch["_actores_norm"] = df_ch[col_actores_ch].apply(normalizar)

In [69]:
REGLAS_CHUBUT = {
    "docentes": r"docente|auxiliar|estudiante|educacion",
    "gremios_estatales": r"sisap|ate\b|soeme|ema\b|cose|concejo deliberante|espacios verdes|salud|traumatolog|cuidadora|personal de la salud|magistratura",
    "sindicatos_privados": r"camionero|estibador|uta\b|uocra|pesquero|cereza|trabajador(es)? de|azucar|aoma|minero",
    "organizaciones_sociales": r"organizacion(es)? social|movimiento(s)? social|mte|madres en lucha|grupos sociales",
    "comunidades_indigenas": r"originari|indigena|mapuche|comunidad(es)?|malon",
    "vecinos": r"vecino|autoconvocad|rodanter|casilla[s]? rodante|feriante|familia|padre",
    "colectivos_ambientales": r"antiminer|asambleist|ambiental|no a la mina",
    "colectivos_especificos": r"vivas y libres|cannabicultor|multisectorial",
    "partidos_politicos": r"militante|frente[s]?|madernista|partido|chubut al frente|pj\b|por trelew",
    "policias_actores": r"policia|cobipol|consejo de bienestar",
    "publico_general": r"manifestante|integrante[s]?|chubutense|pueblo|sociedad civil|vecinos en general",
    "colectivos_violentos": r"encapuchad|vandalo|turba|revoltoso|inadaptad",
    "otros": r"^trabajador(es)?$|dueños|profesionales|trabajadores independientes",
}

In [70]:
for var, patron in REGLAS_CHUBUT.items():
    df_ch[var] = df_ch["_actores_norm"].str.contains(patron, regex=True, na=False).astype(int)

C:\Users\Federico\AppData\Local\Temp\ipykernel_15840\1082084789.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_ch[var] = df_ch["_actores_norm"].str.contains(patron, regex=True, na=False).astype(int)
C:\Users\Federico\AppData\Local\Temp\ipykernel_15840\1082084789.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_ch[var] = df_ch["_actores_norm"].str.contains(patron, regex=True, na=False).astype(int)
C:\Users\Federico\AppData\Local\Temp\ipykernel_15840\1082084789.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_ch[var] = df_ch["_actores_norm"].str.contains(patron, regex=True, na=False).astype(int)
C:\Users\Federico\AppData\Local\Temp\ipykernel_15840\1082084789.py:2: UserWarning: This pattern is interpreted a

In [76]:
col_actores_ch = list(REGLAS_CHUBUT.keys())
df_ch["_tiene_alguno"] = df_ch[col_actores_ch].sum(axis=1)

print("=" * 60)
print("DISTRIBUCIÓN DE CATEGORÍAS — CHUBUT")
print("=" * 60)
for col in col_actores_ch:
    n = df_ch[col].sum()
    pct = df_ch[col].mean() * 100
    print(f"{col}: {n} eventos ({pct:.1f}%)")

sin_clasificar = df_ch[df_ch["_tiene_alguno"] == 0]
print(f"\n⚠ Filas sin clasificar: {len(sin_clasificar)}")
if len(sin_clasificar) > 0:
    print(sin_clasificar[col_actores_ch].to_string(index=False))

df_ch.drop(columns=["_actores_norm", "_tiene_alguno"])
# print("\n✓ Guardado: eventos_chubut_actores_clasificados.xlsx")

DISTRIBUCIÓN DE CATEGORÍAS — CHUBUT
docentes: 2 eventos (1.7%)
gremios_estatales: 18 eventos (14.9%)
sindicatos_privados: 14 eventos (11.6%)
organizaciones_sociales: 5 eventos (4.1%)
comunidades_indigenas: 6 eventos (5.0%)
vecinos: 25 eventos (20.7%)
colectivos_ambientales: 16 eventos (13.2%)
colectivos_especificos: 2 eventos (1.7%)
partidos_politicos: 5 eventos (4.1%)
policias_actores: 9 eventos (7.4%)
publico_general: 13 eventos (10.7%)
colectivos_violentos: 6 eventos (5.0%)
otros: 1 eventos (0.8%)

⚠ Filas sin clasificar: 20
 docentes  gremios_estatales  sindicatos_privados  organizaciones_sociales  comunidades_indigenas  vecinos  colectivos_ambientales  colectivos_especificos  partidos_politicos  policias_actores  publico_general  colectivos_violentos  otros
        0                  0                    0                        0                      0        0                       0                       0                   0                 0                0                  

,noticia_id,link,fecha_publicacion,titulo,localidad,actores_participantes,demanda_principal,tipo_accion,tipo_accion_detalle,disrupcion,...,otros,demanda_salarial,demanda_laboral,demanda_ambiental_territorial,demanda_politica_legal,demanda_ddhh,demanda_servicios,demanda_derechos_especificos,demanda_institucional,demanda_conmemoracion
0,2021103100,https://www.elchubut.com.ar/regionales/2021-10...,2021103100,trabajo a reglamento de personal de la salud,Chubut,Sindicato de la Salud Pública,Pago de deuda de paritarias y retroactivos,Paro,trabajo a reglamento,1,...,0,1,0,0,0,0,0,0,0,0
2,2021110100,https://www.elchubut.com.ar/regionales/2021-11...,2021110100,alumnos recibieron tarea con material antimine...,No especificado,No especificado,No especificado,No especificado,No especificado,99,...,0,0,0,0,0,0,0,0,0,0
3,2021110200,https://www.elchubut.com.ar/regionales/2021-11...,2021110200,estatales cortaron ruta 7 y se reunieron con d...,Rawson,Auxiliares de la educación,falta de convocatoria a paritarias salariales,Corte,Ruta provincial Nº 7,1,...,0,1,0,0,0,0,0,0,0,0
5,2021110202,https://www.elchubut.com.ar/esquel/2021-11-2-2...,2021110202,perez catan viene a esquel a reunirse con mapu...,Esquel,Miembros de comunidades originarias,solución a la titularidad de tierras,Toma de edificio,ocupación del edificio del IAC,1,...,0,0,0,1,0,0,0,0,0,0
6,2021110202,https://www.elchubut.com.ar/esquel/2021-11-2-2...,2021110202,perez catan viene a esquel a reunirse con mapu...,Esquel,Miembros de comunidades originarias,No especificado,Toma de edificio,ocupación en la Secretaría de Bosques,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176,2022010901,https://www.elchubut.com.ar/regionales/2022-1-...,2022010901,quedó libre el último detenido por los daños a...,No especificado,No especificado,No especificado,No especificado,No especificado,99,...,0,0,0,0,0,0,0,0,0,0
178,2022011101,https://www.elchubut.com.ar/regionales/2022-1-...,2022011101,tendrían que demoler al menos una parte de la ...,Rawson,No especificado,Protesta contra la zonificación minera,Manifestación,Incendio provocado,1,...,0,0,0,1,0,0,0,0,0,0
180,2022011600,https://www.elchubut.com.ar/regionales/2022-1-...,2022011600,de puño y letra: lío político con el uso del r...,Chubut,chubutenses,No a la mina / cuidado del agua,Marcha,marchas pacíficas,0,...,0,0,0,1,0,0,0,0,0,0
183,2022012301,https://www.elchubut.com.ar/regionales/2022-1-...,2022012301,de puño y letra: el «año político» empieza a a...,Rawson,Madernistas,No especificado,Acto,destrozos a la Casa de Gobierno / incendio a E...,1,...,0,0,0,0,0,0,0,0,0,0


In [72]:
sin_clasificar = df_ch[df_ch["_tiene_alguno"] == 0]
print(f"Filas sin clasificar: {len(sin_clasificar)}\n")
print(sin_clasificar[["noticia_id", "actores_participantes"]].to_string(index=False))

Filas sin clasificar: 20

 noticia_id                    actores_participantes
 2021110100                          No especificado
 2021111900                          No especificado
 2021112100                          No especificado
 2021121904                          No especificado
 2021121910 Intendente de Trelew / Fieles seguidores
 2021122100                          No especificado
 2021122103                          No especificado
 2021122105                          No especificado
 2021122106                          No especificado
 2021122107                          No especificado
 2021122113                          No especificado
 2021122201                          No especificado
 2021122701                          No especificado
 2021122703                                Imputados
 2021122800                          No especificado
 2022010201                          No especificado
 2022010601                          No especificado
 2022010901         

In [73]:
REGLAS_DEMANDAS = {
    "demanda_salarial": (
        r"salarial|salario|paritaria|aumento|retroactivo|recategoriz|"
        r"recomposicion|sueldo|piso salarial|haberes|deuda.*paritaria|"
        r"deuda.*salarial|incremento salarial|exigir pago"
    ),
    "demanda_laboral": (
        r"condicion(es)? laboral|convenio colectivo|insalubridad|indumentaria|"
        r"higiene|seguridad laboral|elemento[s]? de|despid|reincorpora|despedida|"
        r"contratacion|herramienta[s]?|maquinaria|maltrato|reconocimiento laboral"
    ),
    "demanda_ambiental_territorial": (
        r"mineri|minera|megamineri|zonificacion minera|no a la mina|"
        r"agua|territorio|tierra|desalojo|titularidad|propiedad comunitaria|"
        r"explotacion minera|no a la megamineria"
    ),
    "demanda_politica_legal": (
        r"reforma|constituc|contra.*ley|contra.*gobierno|derogacion|"
        r"veto|ley de pesca|diversificacion productiva|"
        r"contra reforma|prohibicion|restringe"
    ),
    "demanda_ddhh": (
        r"justicia|represion|detencion|femicid|ddhh|derechos humanos|"
        r"repudio|liberacion|detenid|violencia de genero|"
        r"contra las detenciones|apagon del terror|desaparecid|muerte|internar|repudiar.*atentado|repudio.*atentado"
    ),
    "demanda_servicios": (
        r"luz|agua potable|recoleccion|transporte|energia|electricidad|"
        r"suministro|tarifa|servicio[s]?"
    ),
    "demanda_derechos_especificos": (
        r"lgbtiq|lgbt|drag queen|colectivo|mujer|genero|cannabis|"
        r"pase sanitario|derecho[s]? de las mujeres|diversidad"
    ),
    "demanda_institucional": (
        r"elecciones|renuncia|investigacion|amparo|proceso[s]? judicial|"
        r"carta organica|designacion|asuncion|imputad|expediente|"
        r"jueces|magistratura|fiscal|repatentar|casilla.*rodante|homologacion|exigencia.*circular|casilla.*rodante"
    ),
    "demanda_conmemoracion": (
        r"conmemoracion|apagon del terror|aniversario|"
        r"memoria|40 marcha|malon de la paz"
    ),
}

In [74]:
def clasificar_demandas(df, col_demanda):
    df["_demanda_norm"] = df[col_demanda].apply(normalizar)
    
    for var, patron in REGLAS_DEMANDAS.items():
        df[var] = df["_demanda_norm"].str.contains(patron, regex=True, na=False).astype(int)
    
    cols = list(REGLAS_DEMANDAS.keys())
    df["_tiene_alguno"] = df[cols].sum(axis=1)
    
    # Reporte
    print("=" * 60)
    print("DISTRIBUCIÓN DE DEMANDAS")
    print("=" * 60)
    for col in cols:
        n = df[col].sum()
        pct = df[col].mean() * 100
        print(f"{col}: {n} ({pct:.1f}%)")
    
    # Sin clasificar (excluir No aplica, Error, No especificado)
    mask_no_esp = df["_demanda_norm"].isin(["no aplica", "error", "no especificado", ""])
    sin_clasificar = df[(df["_tiene_alguno"] == 0) & (~mask_no_esp)]
    print(f"\n⚠ Sin clasificar: {len(sin_clasificar)}")
    if len(sin_clasificar) > 0:
        print(sin_clasificar[[df.columns[0], col_demanda]].head(20).to_string(index=False))
    
    return df.drop(columns=["_demanda_norm", "_tiene_alguno"])

In [75]:
df_ch= clasificar_demandas(df_ch, "demanda_principal")

DISTRIBUCIÓN DE DEMANDAS
demanda_salarial: 28 (23.1%)
demanda_laboral: 8 (6.6%)
demanda_ambiental_territorial: 40 (33.1%)
demanda_politica_legal: 13 (10.7%)
demanda_ddhh: 12 (9.9%)
demanda_servicios: 3 (2.5%)
demanda_derechos_especificos: 5 (4.1%)
demanda_institucional: 6 (5.0%)
demanda_conmemoracion: 0 (0.0%)

⚠ Sin clasificar: 2
 noticia_id                         demanda_principal
 2021111200          Bloquear descarga de langostinos
 2021121901 Ataques incendiarios a edificios públicos


C:\Users\Federico\AppData\Local\Temp\ipykernel_15840\358751638.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[var] = df["_demanda_norm"].str.contains(patron, regex=True, na=False).astype(int)


In [79]:
df_ch.to_excel("../data/processed/eventos_chubut_limpio.xlsx", index=False)